# Arez — Devil's Advocate
**Model:** Mistral Small 3.1 24B Instruct (4-bit quantized)  
**Hardware:** Colab Pro + A100 (40GB VRAM)  
**Cách dùng:** Chạy Cell 1 → 2 → 3 theo thứ tự. Sau đó chỉ dùng Cell 3.

---

In [ ]:
# ════════════════════════════════════════
# CELL 1 — CÀI THƯ VIỆN
# Chạy 1 lần duy nhất. ~2–3 phút.
# ════════════════════════════════════════

import subprocess
import sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('Đang cài thư viện...')
install('transformers>=4.40.0')
install('bitsandbytes>=0.43.0')
install('accelerate>=0.27.0')
install('huggingface_hub')
install('torch')

print('✅ Cài xong.')

In [ ]:
# ════════════════════════════════════════
# CELL 2 — LOAD MODEL
# Chạy 1 lần duy nhất. ~4–6 phút download + load.
# ════════════════════════════════════════

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = 'mistralai/Mistral-Small-3.1-24B-Instruct-2503'

# Kiểm tra GPU
if not torch.cuda.is_available():
    raise RuntimeError('Không tìm thấy GPU. Vào Runtime > Change runtime type > GPU (A100).')

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB')

# Cấu hình 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

print(f'Đang load tokenizer từ {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print('Đang load model (4-bit)... (~4–6 phút lần đầu)')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
model.eval()

print('✅ Model sẵn sàng.')

# ── SYSTEM PROMPT ──────────────────────
SYSTEM_PROMPT = """Bạn là Arez — một AI đóng vai Devil's Advocate chuyên nghiệp.

Nhiệm vụ của bạn:
- Phản biện mọi luận điểm, quyết định, hoặc kế hoạch mà người dùng đưa ra
- Tìm ra điểm yếu, rủi ro ẩn, giả định sai, hoặc góc nhìn bị bỏ sót
- Đặt câu hỏi sắc bén và trực tiếp
- Không đồng ý một cách dễ dàng — luôn tìm lý do để challenge
- Trả lời bằng tiếng Việt trừ khi được yêu cầu khác
- Ngắn gọn, sắc bén, đi thẳng vào vấn đề — không dài dòng

Bạn KHÔNG phải tìm đồng thuận. Vai trò của bạn là làm cho lập luận của người dùng phải vững hơn."""

# Lưu lịch sử hội thoại
conversation_history = []

In [ ]:
# ════════════════════════════════════════
# CELL 3 — CHAT VỚI AREZ
# Chạy lại cell này mỗi lần muốn nhập câu mới.
# Lịch sử hội thoại được giữ nguyên.
# Gõ 'reset' để bắt đầu cuộc hội thoại mới.
# Gõ 'quit' để thoát.
# ════════════════════════════════════════

def generate_response(messages, max_new_tokens=512):
    """Generate a response from the model."""
    # Apply chat template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors='pt').to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Chỉ lấy phần generated (bỏ phần input)
    new_ids = output_ids[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()


# ── MAIN CHAT LOOP ──────────────────────
print('=' * 50)
print('AREZ — DEVIL\'S ADVOCATE')
print('Gõ "reset" để bắt đầu lại | "quit" để thoát')
print('=' * 50)
print()

# Khởi tạo messages với system prompt
if not conversation_history:
    conversation_history = [{'role': 'system', 'content': SYSTEM_PROMPT}]

while True:
    try:
        user_input = input('Anh: ').strip()
    except EOFError:
        break

    if not user_input:
        continue

    if user_input.lower() == 'quit':
        print('Đã thoát.')
        break

    if user_input.lower() == 'reset':
        conversation_history = [{'role': 'system', 'content': SYSTEM_PROMPT}]
        print('[Lịch sử đã reset. Bắt đầu cuộc hội thoại mới.]\n')
        continue

    # Thêm tin nhắn người dùng
    conversation_history.append({'role': 'user', 'content': user_input})

    print('Arez: ', end='', flush=True)

    # Generate response
    response = generate_response(conversation_history)

    print(response)
    print()

    # Lưu response vào lịch sử
    conversation_history.append({'role': 'assistant', 'content': response})